In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import spacy
import tqdm
import time
from scipy import sparse
from googletrans import Translator


### Pré traitement des données 

In [6]:
df = pd.read_csv('../data/projects_data.csv')

In [7]:
df.head()

,project_name,link,description,composants
0,Solve a Sudoku Puzzle With a CNC Machine,https://www.instructables.com/Solve-a-Sudoku-P...,I created my first Sudoku robot in 2013. It wa...,Consistent with the Introduction's generic des...
1,OpenCycloid - 3D-printed Open-Source Robotic A...,https://www.instructables.com/OpenCycloid-3D-p...,OpenCycloid is a 3D-printed open-source roboti...,Tools needle nose pliers Allen wrench set sold...
2,E-GOR the Retro Robot,https://www.instructables.com/E-GOR-the-Retro-...,"This is E-G0R, a retro styled robot that can b...",Tank Tread Link Egor Parts Link (x4)150 kg/cm...
3,"A Robotic Arm, Based on Mg90s Servos and 28BYJ...",https://www.instructables.com/A-Robotic-Arm-Ba...,"Hi! I’m Wiktor, a 13-year-old electronics, rob...",Base compartment x 1 Base lid x 1 Left gripper...
4,The Smart Fish. Programmable Robot,https://www.instructables.com/The-Smart-Fish-P...,The Smart Fishis a programmable robot based on...,"Just for the curious, I will say that I had th..."


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1509 entries, 0 to 1508
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   project_name  1505 non-null   object
 1   link          1509 non-null   object
 2   description   1184 non-null   object
 3   composants    1174 non-null   object
dtypes: object(4)
memory usage: 47.3+ KB


In [9]:
df.isna().sum()

project_name      4
link              0
description     325
composants      335
dtype: int64

On a:
- *325* projets sans descriptions
- *334* projets dont les composants pour la concéption ne sont pas indiqués.

Je vais tous simplement supprimer les lignes sans l'un de ces attributs car on ne pourra pas récommender à une personne un projet si on ne sait même pas de quoi parle le projet...

In [10]:
df_cleaned = df.dropna(subset=['composants','description'])
df_cleaned.isna().sum()

project_name    0
link            0
description     0
composants      0
dtype: int64

**Je vais traduire les textes en Français pour la suite de l'analyse**

In [18]:
def translate_text(text):
    translator = Translator()
    try:
        return translator.translate(text, src='en', dest='fr').text
    except:
        time.sleep(1)
        return text

def translate_column(df, column_name, batch_size=50):
    """Traduit une colonne par lots"""
    translated = []
    for i in tqdm.tqdm((range(0, len(df), batch_size))):
        batch = df[column_name].iloc[i:i+batch_size] 
        translated.extend([translate_text(text) for text in batch])
        time.sleep(2)  # Délai entre les lots
        
    return translated


In [12]:
tqdm.tqdm.pandas()

In [20]:
# Pour eviter les warning 
df_final = df_cleaned.copy()
df_final.loc[:, 'project_name_fr'] = translate_column(df_final, 'project_name')

  0%|          | 0/23 [00:00<?, ?it/s]

100%|██████████| 23/23 [03:35<00:00,  9.38s/it]


In [22]:
df_final.loc[:, 'composants_fr'] = translate_column(df_final, 'composants')
df_final.loc[:, 'description_fr'] = translate_column(df_final, 'description')

100%|██████████| 23/23 [04:52<00:00, 12.71s/it]


In [22]:
print(df_final.loc[0, 'composants_fr'])

Consistent with the Introduction's generic description of the Sudoku robot, I provide generic component names for my design for the robot. In Step 1, I identify the real components that I used in my implementation. If you create a different design even this generic list could change. Microcomputer Camera Tablet with Sudoku app Vertical and horizontal parts Camera holder Fasteners Stylus for tapping on the tablet Soldering iron Hex wrench Screw drivers


**Sauvegarde du fichier**

In [25]:
df_final.to_csv('../data/projects_translated_fr.csv', index=False)

In [14]:
df_final = pd.read_csv('../data/projects_translated_fr.csv')

In [26]:
df_final.head()

,project_name,link,description,composants,project_name_fr,composants_fr,description_fr
0,Solve a Sudoku Puzzle With a CNC Machine,https://www.instructables.com/Solve-a-Sudoku-P...,I created my first Sudoku robot in 2013. It wa...,Consistent with the Introduction's generic des...,Résolvez un puzzle Sudoku avec une machine CNC,Conformément à la description générique du rob...,J'ai créé mon premier robot Sudoku en 2013. C'...
1,OpenCycloid - 3D-printed Open-Source Robotic A...,https://www.instructables.com/OpenCycloid-3D-p...,OpenCycloid is a 3D-printed open-source roboti...,Tools needle nose pliers Allen wrench set sold...,OpenCycloid - Actionneur robotique open source...,Outils pince à bec effilé jeu de clés Allen fe...,OpenCycloid est un actionneur robotique open s...
2,E-GOR the Retro Robot,https://www.instructables.com/E-GOR-the-Retro-...,"This is E-G0R, a retro styled robot that can b...",Tank Tread Link Egor Parts Link (x4)150 kg/cm...,E-GOR le robot rétro,Tank Tread Link Egor Parts Link (x4)150 kg/cm ...,"Il s'agit d'E-G0R, un robot de style rétro qui..."
3,"A Robotic Arm, Based on Mg90s Servos and 28BYJ...",https://www.instructables.com/A-Robotic-Arm-Ba...,"Hi! I’m Wiktor, a 13-year-old electronics, rob...",Base compartment x 1 Base lid x 1 Left gripper...,"Un bras robotique, basé sur des servos Mg90s e...",Compartiment de base x 1 Couvercle de base x 1...,"Salut! Je m'appelle Wiktor, un passionné d'éle..."
4,The Smart Fish. Programmable Robot,https://www.instructables.com/The-Smart-Fish-P...,The Smart Fishis a programmable robot based on...,"Just for the curious, I will say that I had th...",Le poisson intelligent. Robot programmable,"Juste pour les curieux, je dirai que j'ai eu l...",Le Smart Fish est un robot programmable basé s...


In [27]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1130 entries, 0 to 1443
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   project_name     1130 non-null   object
 1   link             1130 non-null   object
 2   description      1130 non-null   object
 3   composants       1130 non-null   object
 4   project_name_fr  1130 non-null   object
 5   composants_fr    1130 non-null   object
 6   description_fr   1130 non-null   object
dtypes: object(7)
memory usage: 70.6+ KB


On se retrouve avec un seul projet sans composants, je vais la supprimer par la suite

In [28]:
df_final.dropna(inplace=True)

**Suppression des colonnes de base et conversion des textes en minuscules**

In [29]:
df_final = df_final.drop(columns=['project_name','description','composants'])

In [30]:
for col in df_final.columns:
    if col != 'link':
        df_final[col] = df_final[col].str.lower()

**Tokenisation et lemmatisation (peut-être) de la colonne description**

In [33]:
nlp = spacy.load("fr_core_news_sm")

# Traiter les textes en lot
def preprocess_texts_batch(texts, batch_size=100):
    docs = nlp.pipe(texts, batch_size=batch_size, disable=["ner", "parser"])
    processed_texts = [
        " ".join(token.lemma_ for token in doc if not token.is_stop and not token.is_punct)
        for doc in docs
    ]
    return processed_texts

In [34]:
df_final['description_preprocessed'] = preprocess_texts_batch(df_final['description_fr'])

In [35]:
# Sauvegarde le dataframe final
df_final.to_csv('../data/projects_translated_fr.csv',index=False)

### Préparation du système de recommandation

In [36]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [37]:
# Initialisation du vectoriseur TF-IDF
vectorizer = TfidfVectorizer()
# On va appliquer TF-IDF sur les descriptions prétraitées
tfidf_matrix = vectorizer.fit_transform(df_final['description_preprocessed'])

In [38]:
import pickle

with open("../src/features/tfidf_model_project.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

In [39]:
sparse.save_npz('../src/features/tfidf_description_project_matrix.npz',tfidf_matrix)

In [40]:
# Fonction pour calculer les similarités
def find_similar_descriptions(user_input, top_n=5):
    """
    Trouver les descriptions les plus similaires à partir de l'entrée utilisateur.
    Paramètres :
        user_input (str) : Description donnée par l'utilisateur.
        top_n (int) : Nombre de résultats les plus similaires à retourner.
    Retourne :
        DataFrame avec les descriptions et scores de similarité.
    """
    # pré-traitement de l'entrée de l'utilisateur
    user_vec = vectorizer.transform([user_input])
    # similarités cosinus 
    similarities = cosine_similarity(user_vec, tfidf_matrix).flatten()
    # les indices des descriptions les plus similaires
    top_indices = similarities.argsort()[::-1][:top_n]
    # enfin, on retourne les descriptions correspondantes avec les scores
    results = pd.DataFrame({
        'description': df_final.loc[top_indices, 'description_fr'],
        'similarity_score': similarities[top_indices],
        'nom_projet': df_final.loc[top_indices, 'project_name_fr'],
        'composants': df_final.loc[top_indices, 'composants_fr']
    })
    return results


In [41]:
user_description = "Je veux construire un robot suiveur de ligne"
result = find_similar_descriptions(user_description, top_n=3)

In [42]:
result

,description,similarity_score,nom_projet,composants
187,"ce qui est bien avec mon livrehomemade robot, ...",0.281167,robots faits maison livre robot,matériaux : (x1)livre de robots faits maison (...
100,une idée de petite arène robotique portable po...,0.250535,arène de robots portable pour les robots l0cost,"2 montants de cloison de 2,4 m ou 8 pieds (j'a..."
922,bonjour tout le monde! je vais décrire comment...,0.247545,voiture commandée par bouton-poussoir avec bra...,"quatre boîtes en plastique (1 pour la voiture,..."
